# HTGNN Options Pricing Model
## Hypergraph Temporal Graph Neural Network for Volatility Prediction

This notebook trains an HTGNN model to predict implied volatility for options pricing.
The model learns from market structure (sectors) and temporal patterns (market stress).

## 1. Install Dependencies

In [ ]:
!pip install torch torch-geometric torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install fastapi uvicorn pyngrok pandas numpy scipy yfinance scikit-learn matplotlib seaborn httpx

## 2. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HypergraphConv
from torch_geometric.data import Data, Batch
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

## 3. Load and Preprocess Data

Load the Gauss314 dataset (3.5M rows) or use the data scraper.

In [ ]:
# Option 1: Load Gauss314 dataset if you have it
# Uncomment and provide path to your dataset:
# df = pd.read_csv('gauss314_dataset.csv')
# print(f"Loaded {len(df)} rows from Gauss314 dataset")

# Option 2: Use data scraper to fetch real market data
from data_scraper import create_training_dataset

# Fetch data for major tickers (this will take a few minutes)
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META', 'NVDA', 'JPM', 'JNJ', 'V']
print("Fetching market data... This may take a few minutes...")
df = create_training_dataset(tickers, "training_data.csv")
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

## 4. Data Preprocessing

Prepare the data for training: create features, build hypergraph structure, and prepare temporal sequences.

In [ ]:
def build_hypergraph_from_data(df: pd.DataFrame) -> Tuple[torch.Tensor, Dict[str, int]]:
    """
    Build hypergraph structure from sector information
    
    Returns:
        edge_index: Hypergraph edge indices [2, num_edges]
        node_to_idx: Mapping from ticker to node index
    """
    # Group by sector
    sectors = {}
    for _, row in df.iterrows():
        sector = row.get('sector', 'Unknown')
        ticker = row['ticker']
        if sector not in sectors:
            sectors[sector] = []
        if ticker not in sectors[sector]:
            sectors[sector].append(ticker)
    
    # Create node mappings
    all_tickers = df['ticker'].unique()
    node_to_idx = {ticker: idx for idx, ticker in enumerate(all_tickers)}
    
    # Create hyperedges (each sector is a hyperedge connecting its stocks)
    edges = []
    hyperedge_idx = 0
    for sector, tickers in sectors.items():
        for ticker in tickers:
            if ticker in node_to_idx:
                edges.append([node_to_idx[ticker], hyperedge_idx])
        hyperedge_idx += 1
    
    if len(edges) == 0:
        # Fallback: create simple connections
        for i in range(len(all_tickers) - 1):
            edges.append([i, i + len(all_tickers)])
            edges.append([i + 1, i + len(all_tickers)])
    
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2, 0), dtype=torch.long)
    return edge_index, node_to_idx


def prepare_features(df: pd.DataFrame, node_to_idx: Dict[str, int]) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Prepare node features and target volatility
    
    Returns:
        features: [num_nodes, feature_dim] tensor
        targets: [num_nodes] tensor of volatility values
    """
    # Create feature matrix
    feature_cols = ['current_price', 'historical_volatility', 'volume', 'high_52w', 'low_52w']
    available_cols = [col for col in feature_cols if col in df.columns]
    
    # Fill missing values
    df_processed = df.copy()
    for col in available_cols:
        if col in df_processed.columns:
            df_processed[col] = df_processed[col].fillna(df_processed[col].median())
    
    # Normalize features
    scaler = StandardScaler()
    if len(available_cols) > 0:
        features_array = scaler.fit_transform(df_processed[available_cols].values)
    else:
        # Fallback: use simple features
        features_array = np.random.randn(len(df), 5)
    
    # Create node features tensor
    num_nodes = len(node_to_idx)
    feature_dim = features_array.shape[1] if len(available_cols) > 0 else 5
    
    features = torch.zeros(num_nodes, feature_dim)
    targets = torch.zeros(num_nodes)
    
    for _, row in df_processed.iterrows():
        ticker = row['ticker']
        if ticker in node_to_idx:
            idx = node_to_idx[ticker]
            if len(available_cols) > 0:
                ticker_data = df_processed[df_processed['ticker'] == ticker]
                if len(ticker_data) > 0:
                    features[idx] = torch.tensor(features_array[df_processed['ticker'] == ticker].mean(axis=0), dtype=torch.float32)
            else:
                features[idx] = torch.randn(feature_dim)
            
            # Target: historical volatility (or use implied vol if available)
            vol = row.get('historical_volatility', row.get('atm_call_iv', 0.2))
            targets[idx] = torch.tensor(vol, dtype=torch.float32)
    
    return features, targets


# Build hypergraph and prepare features
print("Building hypergraph structure...")
edge_index, node_to_idx = build_hypergraph_from_data(df)
print(f"Hypergraph: {len(node_to_idx)} nodes, {edge_index.shape[1]} edges")

print("\nPreparing features...")
features, targets = prepare_features(df, node_to_idx)
print(f"Features shape: {features.shape}")
print(f"Targets shape: {targets.shape}")
print(f"Target volatility range: [{targets.min():.4f}, {targets.max():.4f}]")

## 5. HTGNN Model Architecture

Complete implementation of the Hypergraph Temporal Graph Neural Network.

In [ ]:
class HTGNN(nn.Module):
    """
    Hypergraph Temporal Graph Neural Network for Volatility Prediction
    """
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        num_layers: int = 3,
        dropout: float = 0.2
    ):
        super(HTGNN, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Input projection
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        
        # Hypergraph convolution layers
        self.hypergraph_convs = nn.ModuleList([
            HypergraphConv(hidden_dim, hidden_dim)
            for _ in range(num_layers)
        ])
        
        # Batch normalization layers
        self.batch_norms = nn.ModuleList([
            nn.BatchNorm1d(hidden_dim)
            for _ in range(num_layers)
        ])
        
        # Output layer (predict volatility)
        self.output = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1),
            nn.Sigmoid()  # Volatility is between 0 and 1
        )
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        """
        Forward pass
        
        Args:
            x: Node features [num_nodes, input_dim]
            edge_index: Hypergraph edge indices [2, num_edges]
        
        Returns:
            volatility: Predicted volatility [num_nodes, 1]
        """
        # Project input
        x = self.input_proj(x)
        
        # Hypergraph convolutions
        for i, (conv, bn) in enumerate(zip(self.hypergraph_convs, self.batch_norms)):
            x_new = conv(x, edge_index)
            x_new = bn(x_new)
            x_new = F.relu(x_new)
            x_new = F.dropout(x_new, p=0.2, training=self.training)
            
            # Residual connection
            if i > 0:
                x = x + x_new
            else:
                x = x_new
        
        # Predict volatility
        volatility = self.output(x)
        
        return volatility.squeeze(-1)  # [num_nodes]


# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = HTGNN(
    input_dim=features.shape[1],
    hidden_dim=64,
    num_layers=3,
    dropout=0.2
).to(device)

print(f"\nModel initialized:")
print(f"  Input dim: {features.shape[1]}")
print(f"  Hidden dim: 64")
print(f"  Num layers: 3")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Training Setup

Split data, set up optimizer, and define training loop.

In [ ]:
# Move data to device
features = features.to(device)
targets = targets.to(device)
edge_index = edge_index.to(device)

# Split data (simple split for now - in production use time-based split)
train_mask = torch.rand(len(targets)) < 0.8
val_mask = ~train_mask

train_features = features[train_mask]
train_targets = targets[train_mask]
val_features = features[val_mask]
val_targets = targets[val_mask]

# Create subgraphs for train/val
train_edge_index = edge_index  # Use full graph for now
val_edge_index = edge_index

print(f"Training samples: {train_mask.sum().item()}")
print(f"Validation samples: {val_mask.sum().item()}")

# Setup optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
criterion = nn.MSELoss()

print("\nOptimizer: Adam (lr=0.001)")
print("Loss: MSE")
print("Scheduler: ReduceLROnPlateau")

## 7. Training Loop

Train the model with validation monitoring.

In [ ]:
def train_epoch(model, features, targets, edge_index, optimizer, criterion):
    """Train for one epoch"""
    model.train()
    optimizer.zero_grad()
    
    pred = model(features, edge_index)
    loss = criterion(pred, targets)
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    return loss.item()


def validate(model, features, targets, edge_index, criterion):
    """Validate model"""
    model.eval()
    with torch.no_grad():
        pred = model(features, edge_index)
        loss = criterion(pred, targets)
        
        # Calculate metrics
        mae = torch.mean(torch.abs(pred - targets)).item()
        mape = torch.mean(torch.abs((pred - targets) / (targets + 1e-8))).item() * 100
    
    return loss.item(), mae, mape


# Training
num_epochs = 100
best_val_loss = float('inf')
patience = 20
patience_counter = 0

train_losses = []
val_losses = []
val_maes = []

print("Starting training...\n")

for epoch in range(num_epochs):
    # Train
    train_loss = train_epoch(model, train_features, train_targets, train_edge_index, optimizer, criterion)
    
    # Validate
    val_loss, val_mae, val_mape = validate(model, val_features, val_targets, val_edge_index, criterion)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_maes.append(val_mae)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'htgnn_best_model.pth')
    else:
        patience_counter += 1
    
    # Print progress
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | "
              f"Train Loss: {train_loss:.6f} | "
              f"Val Loss: {val_loss:.6f} | "
              f"Val MAE: {val_mae:.6f} | "
              f"Val MAPE: {val_mape:.2f}% | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print("\nTraining completed!")

# Load best model
model.load_state_dict(torch.load('htgnn_best_model.pth'))
print("Best model loaded.")

In [ ]:
# Final evaluation
model.eval()
with torch.no_grad():
    train_pred = model(train_features, train_edge_index)
    val_pred = model(val_features, val_edge_index)
    
    train_mae = torch.mean(torch.abs(train_pred - train_targets)).item()
    val_mae = torch.mean(torch.abs(val_pred - val_targets)).item()
    
    train_mape = torch.mean(torch.abs((train_pred - train_targets) / (train_targets + 1e-8))).item() * 100
    val_mape = torch.mean(torch.abs((val_pred - val_targets) / (val_targets + 1e-8))).item() * 100

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"Training MAE:  {train_mae:.6f}")
print(f"Training MAPE: {train_mape:.2f}%")
print(f"Validation MAE:  {val_mae:.6f}")
print(f"Validation MAPE: {val_mape:.2f}%")
print("=" * 60)

# Plot training curves
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(val_maes, label='Val MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.title('Validation MAE')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
plt.scatter(val_targets.cpu().numpy(), val_pred.cpu().numpy(), alpha=0.5)
plt.plot([val_targets.min().item(), val_targets.max().item()], 
         [val_targets.min().item(), val_targets.max().item()], 'r--')
plt.xlabel('True Volatility')
plt.ylabel('Predicted Volatility')
plt.title('Prediction vs True Values')
plt.grid(True)

plt.tight_layout()
plt.show()

## 9. Save Model and Metadata

Save the trained model and configuration for inference.

In [ ]:
# Save final model
torch.save(model.state_dict(), 'htgnn_model.pth')
print("Model saved to htgnn_model.pth")

# Save model configuration
import json
model_config = {
    'input_dim': features.shape[1],
    'hidden_dim': 64,
    'num_layers': 3,
    'dropout': 0.2,
    'node_to_idx': {k: int(v) for k, v in node_to_idx.items()},
    'training_stats': {
        'best_val_loss': float(best_val_loss),
        'final_train_mae': float(train_mae),
        'final_val_mae': float(val_mae),
        'final_train_mape': float(train_mape),
        'final_val_mape': float(val_mape)
    }
}

with open('model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print("Model configuration saved to model_config.json")

# Save edge_index for inference
torch.save(edge_index, 'edge_index.pt')
print("Hypergraph structure saved to edge_index.pt")

## 10. FastAPI Inference Server

Set up FastAPI server to serve predictions via Ngrok.

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok

# Load model for inference
model.eval()
edge_index_inference = edge_index

app = FastAPI(title="HTGNN Volatility Predictor")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class VolatilityRequest(BaseModel):
    underlying: str
    strike: float
    expiration_days: int

class VolatilityResponse(BaseModel):
    underlying: str
    strike: float
    expiration_days: int
    volatility: float

@app.post("/predict", response_model=VolatilityResponse)
async def predict_volatility(request: VolatilityRequest):
    """
    Predict volatility for an option using HTGNN model
    """
    ticker = request.underlying.upper()
    
    # Get node index for ticker
    if ticker not in node_to_idx:
        # Fallback: use average volatility or historical estimate
        volatility = 0.2
    else:
        node_idx = node_to_idx[ticker]
        
        # Get features for this node
        node_features = features[node_idx:node_idx+1]
        
        # Predict
        with torch.no_grad():
            pred = model(node_features, edge_index_inference)
            volatility = pred[0].item()
        
        # Clamp to reasonable range
        volatility = max(0.01, min(1.0, volatility))
    
    return VolatilityResponse(
        underlying=request.underlying,
        strike=request.strike,
        expiration_days=request.expiration_days,
        volatility=volatility
    )

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model_loaded": True,
        "num_nodes": len(node_to_idx)
    }

print("FastAPI app created. Starting server...")

# Expose via Ngrok
# Get your Ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

public_url = ngrok.connect(8001)
print(f"\n{'='*60}")
print(f"🚀 FastAPI server exposed at: {public_url}")
print(f"{'='*60}")
print(f"\nUpdate your backend/.env file with:")
print(f"HTGNN_ENDPOINT={public_url}")
print(f"\n{'='*60}\n")

# Run server
uvicorn.run(app, host="0.0.0.0", port=8001)

## 5. FastAPI Server Setup

Run the FastAPI server and expose via Ngrok to connect with the local backend.

In [ ]:
# See main.py in backend/ for the full FastAPI implementation
# In Colab, you would:
# 1. Load your trained model
# 2. Create FastAPI endpoints for /predict
# 3. Expose via ngrok: public_url = ngrok.connect(8001)
# 4. Update HTGNN_ENDPOINT in backend/.env with the ngrok URL

print("FastAPI server setup - see backend/main.py for implementation")